# 04. Static rule base — 정상 train 기준 99.9% 임계

완전중복 제거 후 세그먼트를 구성하고 정상 전체를 시간순 60/20/20으로 분할한다.
주 평가는 n≥10이며 짧은 세그먼트는 판단 보류로 집계한다. 이상은 파일 라벨 기준이다.
6개 개별 규칙과 NRMSE 단독 / NRMSE+상하부 RMS / 전체 OR를 동일 표본에서 비교한다.
`env_cv` 및 지속성 규칙은 이번 실험에 포함하지 않는다.

임계와 사인·회귀 기준은 정상 train만 사용한다. validation은 중간 점검용이며 이번에는
99.9% 규칙을 고정하고 임계를 재조정하지 않는다. outlier는 학습에 사용하지 않는다.
피처 선정에 이미 outlier를 참고했으므로 독립 외부 검증은 아니다.

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar

DATA_DIR = Path(os.environ.get('KAMP_DATA_DIR', '/mnt/d/data/kamp_ai/press_anomaly_dataset'))
GAP_THR = .15
MIN_N = 10
COVERAGE = .999
SIGNALS = ['AI0_Vibration', 'AI1_Vibration', 'AI2_Current']
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 180)


## 1. 중복 처리와 시간순 분할
센서값·시각·라벨이 모두 같은 완전중복만 첫 행을 남긴다. 동일 시각에 다른 값이 있으면
자동 제거하지 않고 중단한다. gap은 보간하지 않는다. 길이 필터링 전에 분할해 시간 경계를 고정한다.

In [2]:
SEG, TABLE, audit = {}, {}, []
for name, filename in [('normal', 'press_data_normal.csv'), ('outlier', 'outlier_data.csv')]:
    raw = pd.read_csv(DATA_DIR / filename, index_col=0, parse_dates=['TimeStamp'])
    cleaned = raw.drop_duplicates().copy()
    assert not cleaned.TimeStamp.duplicated().any(), '동일 timestamp의 서로 다른 값 확인 필요'
    assert cleaned.TimeStamp.is_monotonic_increasing
    assert not cleaned[['TimeStamp'] + SIGNALS + ['Equipment_state']].isna().any().any()
    assert cleaned.Equipment_state.eq(0 if name == 'normal' else 1).all()
    cleaned['seg_id'] = (cleaned.TimeStamp.diff().dt.total_seconds() > GAP_THR).cumsum()
    SEG[name] = cleaned
    rows = []
    for sid, g in cleaned.groupby('seg_id'):
        row = dict(seg_id=sid, n=len(g), t_start=g.TimeStamp.iloc[0], t_end=g.TimeStamp.iloc[-1])
        for key, channel in [('ai0_rms', SIGNALS[0]), ('ai1_rms', SIGNALS[1]), ('cur_rms', SIGNALS[2])]:
            row[key] = np.sqrt(np.mean(g[channel].to_numpy(float)**2))
        rows.append(row)
    TABLE[name] = pd.DataFrame(rows).sort_values('t_start').reset_index(drop=True)
    audit.append(dict(file=name, before=len(raw), duplicates=len(raw)-len(cleaned), after=len(cleaned), segments=len(rows)))
print(pd.DataFrame(audit).to_string(index=False))
normal = TABLE['normal'].copy()
a, b = int(len(normal)*.6), int(len(normal)*.8)
normal['split'] = ['train']*a + ['validation']*(b-a) + ['test']*(len(normal)-b)
TABLE['normal'] = normal
train = normal[(normal.split == 'train') & (normal.n >= MIN_N)].copy()
assert len(train) > 0
print(normal.groupby('split', sort=False).agg(total=('n', 'size'), eligible=('n', lambda x: (x>=MIN_N).sum())).to_string())
print('outlier:', len(TABLE['outlier']), 'eligible:', (TABLE['outlier'].n>=MIN_N).sum())


   file  before  duplicates  after  segments
 normal   20000           1  19999       599
outlier     600           0    600        21
            total  eligible
split                      
train         359       313
validation    120       108
test          120       109
outlier: 21 eligible: 17


## 2. 정상 기준과 6개 피처
전류의 공통 주파수·중심은 train 중 n≥30에서 추정한다. 주파수 탐색은 기존 정상 대역
1/2.2~1/0.9Hz이며 세그먼트 추정값의 중앙값을 공통 주파수로 사용한다.
공통 주파수로 다시 자유 중심 적합해 중심의 중앙값을 구한다.

NRMSE는 정상 주파수·중심을 고정하고 각 세그먼트의 진폭·위상만 적합한다.
`NRMSE = sqrt(sum((x-pred)^2) / sum((x-mean(x))^2))`.
전류 대비 진동은 단순 비율 대신 정상 train의 선형 회귀 잔차를 사용한다.
상하부 비율은 AI1 RMS / AI0 RMS이다.

In [4]:
def sine_fit(g, frequency, center=None):
    x = g.AI2_Current.to_numpy(float)
    t = (g.TimeStamp-g.TimeStamp.iloc[0]).dt.total_seconds().to_numpy()
    design = np.column_stack([np.cos(2*np.pi*frequency*t), np.sin(2*np.pi*frequency*t)])
    if center is None:
        design = np.column_stack([design, np.ones(len(x))])
        coef = np.linalg.lstsq(design, x, rcond=None)[0]
        pred = design @ coef
    else:
        coef = np.linalg.lstsq(design, x-center, rcond=None)[0]
        pred = center + design @ coef
    ss = np.sum((x-pred)**2)
    total = np.sum((x-x.mean())**2)
    return ss, (np.sqrt(ss/total) if total > 0 else np.nan), coef

reference = train[train.n>=30]
assert len(reference) > 0
frequencies = []
for sid in reference.seg_id:
    g = SEG['normal'][SEG['normal'].seg_id == sid]
    grid = np.linspace(1/2.2, 1/.9, 401)
    best = int(np.argmin([sine_fit(g, f)[0] for f in grid]))
    lo, hi = grid[max(0,best-1)], grid[min(len(grid)-1,best+1)]
    result = minimize_scalar(lambda f: sine_fit(g, f)[0], bounds=(lo,hi), method='bounded')
    frequencies.append(min([grid[best], result.x], key=lambda f: sine_fit(g,f)[0]))
f_reference = float(np.median(frequencies))
center = float(np.median([sine_fit(SEG['normal'][SEG['normal'].seg_id==sid], f_reference)[2][-1]
                          for sid in reference.seg_id]))
regression = {key: np.polyfit(train.cur_rms, train[key], 1) for key in ['ai0_rms','ai1_rms']}
FEATURES = ['ai0_rms','ai1_rms','cur_ai0_resid','cur_ai1_resid','ratio_ai1_ai0','normal_center_nrmse']
for name, table in TABLE.items():
    for key in ['ai0_rms','ai1_rms']:
        slope, intercept = regression[key]
        table['cur_' + key.split('_')[0] + '_resid'] = table[key]-(slope*table.cur_rms+intercept)
    table['ratio_ai1_ai0'] = table.ai1_rms / table.ai0_rms.replace(0,np.nan)
    table['normal_center_nrmse'] = [sine_fit(SEG[name][SEG[name].seg_id==sid], f_reference, center)[1]
                                    for sid in table.seg_id]
print(f'정상 주파수={f_reference:.6f}Hz, 주기={1/f_reference:.6f}s, 중심={center:.6f}, 기준 표본={len(reference)}')
print('회귀 계수 (slope, intercept):', regression)


정상 주파수=0.600039Hz, 주기=1.666559s, 중심=0.555780, 기준 표본=215
회귀 계수 (slope, intercept): {'ai0_rms': array([0.00032309, 0.03274043]), 'ai1_rms': array([ 0.00145313, -0.06514353])}


## 3. 고정 임계와 판정
RMS·NRMSE는 train의 상단 99.9백분위수 초과를 이상으로 본다.
회귀 잔차·상하부 비율은 중앙 99.9% 구간(0.05~99.95백분위수) 밖을 이상으로 본다.
분위수는 선형 보간한다. 표본 수가 작아 극값에 가까운 기준이며 운영 오경보 0.1%를 보장하지 않는다.
OR 조합의 오경보율도 별도로 측정한다. 선택한 피처가 유한하지 않으면 해당 규칙은 판단 보류한다.

In [5]:
train_features = TABLE['normal'].query("split == 'train' and n >= @MIN_N")
TWO_SIDED = {'cur_ai0_resid','cur_ai1_resid','ratio_ai1_ai0'}
thresholds = {}
for feature in FEATURES:
    values = train_features[feature]
    assert np.isfinite(values).all()
    lower = values.quantile((1-COVERAGE)/2) if feature in TWO_SIDED else -np.inf
    upper = values.quantile(1-(1-COVERAGE)/2 if feature in TWO_SIDED else COVERAGE)
    thresholds[feature] = (lower,upper)
print(pd.DataFrame([dict(feature=f, lower=v[0], upper=v[1]) for f,v in thresholds.items()]).to_string(index=False))
RULES = {f:[f] for f in FEATURES}
RULES.update({'nrmse_only':['normal_center_nrmse'],
              'nrmse_rms_OR':['normal_center_nrmse','ai0_rms','ai1_rms'],
              'all_6_OR':FEATURES})
def predict(table, features):
    valid = (table.n>=MIN_N) & np.isfinite(table[features]).all(axis=1)
    alarm = pd.Series(False,index=table.index)
    for f in features:
        lo,hi = thresholds[f]
        alarm |= (table[f]<lo) | (table[f]>hi)
    return valid, alarm & valid


            feature     lower    upper
            ai0_rms      -inf 0.137925
            ai1_rms      -inf 0.215677
      cur_ai0_resid -0.027255 0.069598
      cur_ai1_resid -0.034944 0.065428
      ratio_ai1_ai0  0.473615 2.943657
normal_center_nrmse      -inf 0.093495


## 4. 공통 평가표
정상 validation과 test를 구분해 표시한다. 동일한 outlier 전체 중 n≥10을 각 표에 사용한다.
정상 탐지율은 specificity(TN/(TN+FP)), 이상 탐지율은 recall(TP/(TP+FN))이다.
Balanced Accuracy는 두 비율의 평균이다. 보류 건수는 분모에서 제외하고 별도로 표시한다.
이상 표본의 파일 라벨을 적용하므로 세그먼트별 실제 고장 여부를 확정하는 결과는 아니다.

In [6]:
metrics = []
for split in ['validation','test']:
    nt = TABLE['normal'][TABLE['normal'].split==split]
    ot = TABLE['outlier']
    for rule, features in RULES.items():
        nv,na = predict(nt,features)
        ov,oa = predict(ot,features)
        fp,tp = int(na.sum()),int(oa.sum())
        tn,fn = int(nv.sum())-fp,int(ov.sum())-tp
        specificity = tn/nv.sum() if nv.sum() else np.nan
        recall = tp/ov.sum() if ov.sum() else np.nan
        metrics.append(dict(split=split, rule=rule, TN=tn, FP=fp, TP=tp, FN=fn,
                            normal_deferred=int((~nv).sum()), outlier_deferred=int((~ov).sum()),
                            normal_detection_pct=100*specificity, false_alarm_pct=100*(1-specificity),
                            anomaly_detection_pct=100*recall, miss_pct=100*(1-recall),
                            balanced_acc_pct=50*(specificity+recall)))
METRICS = pd.DataFrame(metrics)
for split in ['validation','test']:
    print('\n',split)
    print(METRICS[METRICS.split==split].round(3).to_string(index=False))



 validation
     split                rule  TN  FP  TP  FN  normal_deferred  outlier_deferred  normal_detection_pct  false_alarm_pct  anomaly_detection_pct  miss_pct  balanced_acc_pct
validation             ai0_rms 108   0  14   3               12                 4               100.000            0.000                 82.353    17.647            91.176
validation             ai1_rms 105   3   9   8               12                 4                97.222            2.778                 52.941    47.059            75.082
validation       cur_ai0_resid 106   2  16   1               12                 4                98.148            1.852                 94.118     5.882            96.133
validation       cur_ai1_resid 108   0  15   2               12                 4               100.000            0.000                 88.235    11.765            94.118
validation       ratio_ai1_ai0 106   2   6  11               12                 4                98.148            1.852       

## 5. 오류 및 길이별 점검
전체 OR의 정상 test 오경보·outlier 미탐과 판단 보류를 출력한다.
각 outlier에서 어떤 규칙이 발동했는지 확인하고 seg3·19·20도 개별 확인한다.

In [ ]:
for name, table in [('normal test', TABLE['normal'].query("split == 'test'")), ('outlier', TABLE['outlier'])]:
    valid,alarm = predict(table,FEATURES)
    error = alarm if name=='normal test' else valid & ~alarm
    print('\n',name,'오류:')
    print(table.loc[error,['seg_id','n']+FEATURES].to_string(index=False))
    print('판단 보류:',table.loc[~valid,['seg_id','n']].to_dict('records'))
out = TABLE['outlier'].copy()
out['triggered_rules'] = [','.join(f for f in FEATURES if np.isfinite(row[f]) and
                          (row[f]<thresholds[f][0] or row[f]>thresholds[f][1]))
                          for _,row in out.iterrows()]
print('\noutlier 규칙별 발동 (n<10은 최종 판단 보류):')
print(out[['seg_id','n','triggered_rules']].to_string(index=False))
length_rows=[]
for label,lower,upper in [('10~19',10,20),('20~29',20,30),('≥30',30,np.inf)]:
    nt=TABLE['normal'].query("split == 'test'")
    ot=TABLE['outlier']
    nt=nt[(nt.n>=lower)&(nt.n<upper)]
    ot=ot[(ot.n>=lower)&(ot.n<upper)]
    nv,na=predict(nt,FEATURES); ov,oa=predict(ot,FEATURES)
    length_rows.append(dict(length=label, normal_count=int(nv.sum()), FP=int(na.sum()),
                            outlier_count=int(ov.sum()), TP=int(oa.sum()), FN=int((ov & ~oa).sum())))
print('\n전체 OR 길이별 건수:')
print(pd.DataFrame(length_rows).to_string(index=False))
assert TABLE['normal'].seg_id.nunique()==599
assert TABLE['outlier'].seg_id.nunique()==21
assert METRICS.query("split == 'test'").normal_deferred.nunique()==1



 normal test 오류:
 seg_id  n  ai0_rms  ai1_rms  cur_ai0_resid  cur_ai1_resid  ratio_ai1_ai0  normal_center_nrmse
    482 50 0.030964 0.038908      -0.029551      -0.020868       1.256549             0.072984
    484 44 0.029287 0.039358      -0.030797      -0.018479       1.343879             0.074646
    488 23 0.032984 0.037080      -0.028704      -0.027972       1.124168             0.074965
    498 50 0.029579 0.038536      -0.030935      -0.021235       1.302812             0.068321
    501 24 0.029984 0.045413      -0.030062      -0.012254       1.514559             0.073723
    511 38 0.030411 0.034705      -0.031201      -0.030002       1.141209             0.069857
    516 40 0.033155 0.041038      -0.027590      -0.019772       1.237739             0.070334
    521 39 0.027855 0.041772      -0.032658      -0.017996       1.499613             0.068813
    523 11 0.040464 0.033251      -0.017443      -0.014794       0.821733             0.096494
    529 11 0.026022 0.048746    

## 해석 시 주의
정상 test가 0건 오경보여도 실제 운영 오경보율 0을 보장하지 않는다.
normal과 outlier의 날짜·운전조건 교락은 그대로 남아 있다.
추후 학습 모델도 동일 전처리·분할·평가 표본을 사용하며, 지도학습에서는 학습에 쓴
outlier를 평가에서 제외하고 공통 미사용 표본으로 비교해야 한다.

## 실행 결과 요약

정상 완전중복 1행을 제거했다(20,000 → 19,999행). 정상 test는 n≥10 109개,
outlier는 n≥10 17개이며 각각 짧은 세그먼트 11개·4개는 판단 보류다.

| 규칙 | 정상 탐지율 | 오경보율 | 이상 탐지율 | 미탐률 | Balanced Accuracy |
|---|---:|---:|---:|---:|---:|
| NRMSE 단독 | 97.25% | 2.75% (3/109) | 100% (17/17) | 0% | 98.62% |
| NRMSE + 상하부 RMS OR | 97.25% | 2.75% (3/109) | 100% (17/17) | 0% | 98.62% |
| 전체 6개 OR | 80.73% | 19.27% (21/109) | 100% (17/17) | 0% | 90.37% |

상부 전류-진동 회귀 잔차의 양측 규칙에서 정상 test 오경보 19건이 발생했다.
전체 OR는 탐지 증가 없이 오경보를 늘렸다. NRMSE 단독은 정상 test seg523·529·585에서
오경보가 발생했다(모두 n=11). 기존 70/30 결과와 달리 60/20/20 train만으로
기준·임계를 추정한 결과이며, 기존 AUC 완전분리가 현재 임계에서 오경보 0을 뜻하지 않는다.

이 결과를 고정된 비교 기준선으로 남기고 ML 비교로 진행한다. test 결과를 보고 임계를
재조정하지 않는다. 오류의 원인은 후속 분석에서 확인한다.